# 🎬 Manhwa Video Studio — Google Colab Edition

Run **Manhwa Video Studio** in Google Colab with hardware acceleration, high-speed cloud vCPUs, and an instant public HTTPS link.

### ⚡ How to Enable GPU in Colab (Optional):
Go to **Runtime** (top menu) → **Change runtime type** → select **T4 GPU** → click **Save**.

*(Note: The studio runs blazing fast on both T4 GPU and standard CPU with our multi-threaded ultrafast engine!)*

## 📦 Step 1: Install Dependencies
Installs FFmpeg and Python libraries.

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q fastapi "uvicorn[standard]" requests pillow numpy pydantic python-multipart

## 📁 Step 2: Get the Project Files from GitHub
Clones your official repository directly into Colab.

In [ ]:
import os
if not os.path.exists('app.py'):
    !git clone https://github.com/mohammedsameeralishaik/manhwa-video-studio.git repo && cp -r repo/* ./ && rm -rf repo
    print("✅ Project code cloned successfully from GitHub!")
else:
    print("✅ Project files (app.py, static/) already present!")

## 🚀 Step 3: Launch Studio with Free Cloudflare Tunnel (1-Click)
Runs the FastAPI server and creates a secure public HTTPS link (`https://xxxx.trycloudflare.com`). **No signup or token required!**

*(Leave this cell running so the web tunnel stays online)*

In [ ]:
import subprocess, time, os, re
from IPython.display import display, HTML

# 1. Download Cloudflare Tunnel binary
if not os.path.exists("cloudflared"):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

# 2. Kill any previous instance
!pkill -f "uvicorn app:app" || true
!pkill -f "cloudflared" || true
time.sleep(1)

# 3. Start Uvicorn backend in background
server_proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)

# 4. Launch Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

print("Starting tunnel... finding public URL...")
public_url = None
start = time.time()
while time.time() - start < 35:
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 60)
    print(f"🎉 STUDIO ONLINE: {public_url}")
    print("=" * 60)
    display(HTML(f'''
    <div style="background: #111827; border: 2px solid #f59e0b; padding: 20px; border-radius: 12px; margin: 15px 0;">
      <h2 style="color: #fbbf24; margin-top: 0;">🎬 Manhwa Video Studio is Live!</h2>
      <p style="color: #d1d5db; font-size: 14px;">Click the button below to open the studio in a new tab:</p>
      <a href="{public_url}" target="_blank" style="background: linear-gradient(135deg, #f59e0b, #d97706); color: #000; padding: 12px 24px; border-radius: 8px; font-weight: bold; text-decoration: none; display: inline-block; font-size: 16px;">
        🚀 Open Manhwa Video Studio
      </a>
    </div>
    '''))
else:
    print("Could not automatically detect the URL. Run '!pkill -f cloudflared' and retry.")